## plots of input data to the model

In [1]:
import pandas as pd
import gdxtools 
import glob
import os
import gams
import numpy as np
import gdxtools as gt
from math import ceil
%matplotlib inline
import plotly.express as px
import plotly

In [2]:
country_map = {'AT' : 'AUT', 'BE' : 'BEL', 'BG' : 'BGR', 'CH' : 'CHE', 'CZ' : 'CZE', 'DE' : 'DEU', 'DK' : 'DNK', 'ES' : 'ESP', 'FI' : 'FIN', 'FR' : 'FRA', 'GB' : 'GBR', 'GR' : 'GRC', 'HR' : 'HRV', 'HU' : 'HUN', 'IE' : 'IRL', 'IT' : 'ITA', 'LU' : 'LUX', 'NL' : 'NLD', 'NO' : 'NOR', 'PL' : 'POL', 'PT' : 'PRT', 'RO' : 'ROU', 'SE' : 'SWE', 'SI' : 'SVN', 'SK' : 'SVK'}

In [3]:
path_plots = "../figures/"
fn_abate = "../../model/data/mac_linear.csv"
fn_gdx = "../../model/scenarios/invest/results/investLP_15.gdx" # standarad input file for remaining data

In [4]:
fn_abate

'../../model/data/mac_linear.csv'

# Initialize GDX

In [5]:
dir_gms = os.getcwd()
ws = gams.GamsWorkspace(dir_gms)

In [6]:
gdx_base = ws.add_database_from_gdx(fn_gdx)

# abatement cost

In [7]:
#load gdx data on load and assign periods for ntc on files
df_abate = pd.read_csv(fn_abate)
df_abate.head()

,region,sector,coefficientLinear,baselineEmissionsMt
0,AT,Agriculture,2270.053121,0.681102
1,AT,EnergieIntensive,160.838482,5.911850
2,AT,IndustryServices,311.415009,5.050427
3,AT,Transport,56.244108,22.704286
4,AT,PrivateHeat,310.010602,4.903078


In [14]:
fig = px.bar(df_abate, x='sector',y='coefficientLinear',color='region',
             barmode='group', log_y=True,
             labels={'coefficientLinear':'Marginal abatement cost [Euro per t CO2]'})
fig.write_image(path_plots+'input_coefficient_abate.pdf')
fig.write_image(path_plots+'input_coefficient_abate.png')
fig.show()

# Investment cost

In [15]:
df_invest = pd.DataFrame(gt.get_symbol_values(gdx_base, "cinv_0", col_names=["tech", "country"], kind="value")).reset_index()
df_invest.head()

,tech,country,Value
0,Solar,AT,37.6
1,Solar,BE,42.2
2,Solar,BG,40.9
3,Solar,CH,42.3
4,Solar,CZ,37.7


In [16]:
df_invest['ISO3'] = df_invest.country.map(country_map)

In [18]:
fig = px.choropleth(df_invest, locations = 'ISO3',
              color = 'Value', 
              facet_col='tech',
              labels={"Value": "€/MWh"},
              color_continuous_scale='plasma',
              scope = 'europe', projection = 'equirectangular')
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_geos(
    lataxis_range=[34,65], lonaxis_range=[-10, 30],
    visible=False, showcountries=True, showland=True)
fig.write_image(path_plots+'input_RES_investment_cost.pdf')
fig.write_image(path_plots+'input_RES_investment_cost.png')
fig.show()

# Private Transport

In [34]:
df_transport_carb_coef = pd.DataFrame(gt.get_symbol_values(gdx_base, "transport_carb_coef", col_names=["country", "tech"], kind="value")).reset_index()
df_transport_carb_coef = df_transport_carb_coef[df_transport_carb_coef.Value != 1000].drop(columns='tech')
df_transport_carb_coef = df_transport_carb_coef.rename(columns={'Value':'carb_coef'})
df_transport_carb_coef.head()

,country,carb_coef
0,AT,1.414016
1,BE,1.608347
2,BG,0.452778
4,CZ,0.822344
5,DE,1.683385


In [35]:
df_transport_ev_load = pd.DataFrame(gt.get_symbol_values(gdx_base, "transport_ev_load", col_names=["country"], kind="value")).reset_index()
df_transport_ev_load = df_transport_ev_load.rename(columns={'Value':'ev_load'})
df_transport_ev_load.head()

,country,ev_load
0,AT,1.870000
1,BE,2.265276
2,BG,0.448995
3,CZ,1.078035
4,DE,2.070624


In [47]:
df_transport_stock = pd.DataFrame(gt.get_symbol_values(gdx_base, "transport_stock", col_names=["country"], kind="value")).reset_index()
df_transport_stock = df_transport_stock.rename(columns={'Value':'stock'})
df_transport_stock['stock'] = df_transport_stock['stock']/1e6
df_transport_stock.head()

,country,stock
0,AT,4.898578
1,BE,5.785447
2,BG,2.770615
3,CZ,5.538222
4,DE,46.474538


In [48]:
df_transport = df_transport_carb_coef.merge(df_transport_ev_load,on='country')
df_transport = df_transport.merge(df_transport_stock,on='country')
df_transport.head()

,country,carb_coef,ev_load,stock
0,AT,1.414016,1.870000,4.898578
1,BE,1.608347,2.265276,5.785447
2,BG,0.452778,0.448995,2.770615
3,CZ,0.822344,1.078035,5.538222
4,DE,1.683385,2.070624,46.474538


In [49]:
df_transport['ISO3'] = df_transport.country.map(country_map)

In [50]:
fig = px.choropleth(df_transport, locations = 'ISO3',
              color = 'carb_coef', 
              labels={"carb_coef": "tons of CO2"},
              color_continuous_scale='PuRd',
              scope = 'europe', projection = 'equirectangular')
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_geos(
    lataxis_range=[34,65], lonaxis_range=[-10, 30],
    visible=False, showcountries=True, showland=True)
fig.write_image(path_plots+'input_transport_carb_coef.pdf')
fig.show()

In [51]:
fig = px.choropleth(df_transport, locations = 'ISO3',
              color = 'ev_load', 
              labels={"ev_load": "MWh"},
              color_continuous_scale='PuRd',
              scope = 'europe', projection = 'equirectangular')
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_geos(
    lataxis_range=[34,65], lonaxis_range=[-10, 30],
    visible=False, showcountries=True, showland=True)
fig.write_image(path_plots+'input_transport_ev_load.pdf')
fig.show()

In [52]:
fig = px.choropleth(df_transport, locations = 'ISO3',
              color = 'stock', 
              labels={"stock": "million cars"},
              color_continuous_scale='PuRd',
              scope = 'europe', projection = 'equirectangular')
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_geos(
    lataxis_range=[34,65], lonaxis_range=[-10, 30],
    visible=False, showcountries=True, showland=True)
fig.write_image(path_plots+'input_transport_stock.pdf')
fig.show()